In [1]:
import spacy
from spacy.matcher import Matcher

nlp = spacy.load("en_core_web_sm")

matcher = Matcher(nlp.vocab)

bedrooms_pattern = [
    [{"LIKE_NUM": True},
     {"LOWER": "bed"},
     {"LOWER": {"IN": ["room", "rooms"]}}],

    [{"LIKE_NUM": True},
     {"LOWER": {"IN": ["bedroom", "bedrooms"]}}]
]
matcher.add("BEDROOMS", bedrooms_pattern)

confirmation_patterns = [
    [{"LOWER": {"IN": ["yes", "yeah", "yep", "yup"]}}],
    [{"LOWER": "sure"}],
    [{"LOWER": {"IN": ["ok", "okay"]}}],
    [{"LOWER": "correct"}],
    [{"LOWER": "sounds"}, {"LOWER": {"IN": ["good", "perfect", "great"]}}],
    [{"LOWER": "that"}, {"LOWER": "sounds"}, {"LOWER": {"IN": ["good", "perfect", "great"]}}],
    [{"LOWER": "that"}, {"LOWER": "works"}],
    [{"LOWER": "that's"}, {"LOWER": "right"}]
]
matcher.add("CONFIRMATION", confirmation_patterns)


In [2]:
import json

with open("homes.json", "r", encoding="utf-8") as f:
    data = json.load(f)

item = -1

for dialog in data:
    item += 1
    turn = -1
    for message in dialog["turns"]:
        turn += 1
        if message["speaker"] != "USER":
            continue
        doc = nlp(message["utterance"])
        matches = matcher(doc)
        for match_id, start, end in matches:
            span = doc[start:end]
            label = nlp.vocab.strings[match_id]
            if label == "CONFIRMATION":
                flag = False
                for in_id, in_start, in_end in matches:
                    if start == in_start and end < in_end:
                        flag = True
                if flag == True: continue
            print(f"Item: {item}, turn: {turn}, {label}: {span.text}")


Item: 0, turn: 2, BEDROOMS: one bed room
Item: 0, turn: 12, CONFIRMATION: Sounds good
Item: 0, turn: 16, CONFIRMATION: Yep
Item: 0, turn: 16, CONFIRMATION: Sounds good
Item: 1, turn: 0, BEDROOMS: one bedroom
Item: 1, turn: 10, CONFIRMATION: sounds good
Item: 1, turn: 12, CONFIRMATION: Yes
Item: 1, turn: 12, CONFIRMATION: that sounds good
Item: 1, turn: 12, CONFIRMATION: sounds good
Item: 1, turn: 14, CONFIRMATION: sounds good
Item: 2, turn: 2, BEDROOMS: one bed room
Item: 2, turn: 6, CONFIRMATION: Yes
Item: 2, turn: 6, CONFIRMATION: sounds good
Item: 2, turn: 8, CONFIRMATION: Yes
Item: 2, turn: 12, CONFIRMATION: Yes
Item: 2, turn: 12, CONFIRMATION: that sounds good
Item: 2, turn: 12, CONFIRMATION: sounds good
Item: 3, turn: 14, CONFIRMATION: Yes
Item: 4, turn: 0, BEDROOMS: 2 bedroom
Item: 4, turn: 8, CONFIRMATION: sounds good
Item: 4, turn: 10, CONFIRMATION: Sure
Item: 4, turn: 14, CONFIRMATION: Yup
Item: 4, turn: 14, CONFIRMATION: sounds good
Item: 5, turn: 2, BEDROOMS: 3 bedrooms
Ite

In [3]:
for dialog in data:
    for message in dialog["turns"]:
        if message["speaker"] != "USER":
            continue
        doc = nlp(message["utterance"])
        for sent in doc.sents:
            for token in sent:
                if token.dep_ == "dobj":
                    dobj = token.text
                    conj = [t.text for t in token.conjuncts]
                    verb = token.head.text
                    print(f"verb: {verb}, dobj: {dobj}, {conj}")


verb: want, dobj: apartment, []
verb: find, dobj: apartments, []
verb: find, dobj: other, []
verb: Give, dobj: number, []
verb: Schedule, dobj: visit, []
verb: found, dobj: options, []
verb: give, dobj: number, []
verb: allow, dobj: pets, []
verb: take, dobj: look, []
verb: Thank, dobj: you, []
verb: thank, dobj: you, []
verb: find, dobj: apartment, []
verb: want, dobj: apartment, []
verb: tell, dobj: me, []
verb: show, dobj: apartments, []
verb: tell, dobj: property, []
verb: have, dobj: sounds, []
verb: have, dobj: sounds, []
verb: find, dobj: apartment, []
verb: find, dobj: one, []
verb: find, dobj: one, []
verb: find, dobj: apartments, []
verb: have, dobj: baths, []
verb: schedule, dobj: visit, []
verb: Thank, dobj: you, []
verb: find, dobj: apartment, []
verb: have, dobj: baths, []
verb: have, dobj: bedrooms, []
verb: like, dobj: apartment, []
verb: find, dobj: apartments, []
verb: schedule, dobj: visit, []
verb: Thank, dobj: you, []
verb: thank, dobj: you, []
verb: find, dobj: ap